In [ ]:
import numpy as np
import pandas as pd
import re
import json
import requests
import concurrent.futures
import yaml
from tqdm import tqdm

# LLM_MODEL = 'llama3.1:8b'
LLM_MODEL = 'qwen2.5:32b'
# LLM_MODEL = 'phi4:14b'
# LLM_MODEL = 'gemma3:1b'
# LLM_MODEL = 'llama3.3:70b'
# LLM_MODEL = 'llama3.2:3b'
# LLM_MODEL = 'deepseek-r1:14b'
# LLM_MODEL = 'qwq:32b'

In [ ]:
import os
os.chdir('../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

In [ ]:
to_search = pd.read_parquet(dataset_config['path_processed'] + 'CN_CN/POST1_not_recognized.parquet')
to_search

In [ ]:
geo_db = pd.DataFrame(to_search.affiliationame.unique(), columns=['affiliationame'])
geo_db

In [ ]:
ollama_base_url = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434").rstrip("/")
generate_url = f"{ollama_base_url}/api/generate"
chat_url = f"{ollama_base_url}/api/chat"

data = {
    "model": LLM_MODEL,
    "prompt": "What is the capital of France?",
    "stream": False  # Set to True for streaming
}

test_response = requests.post(generate_url, json=data)
print(test_response.json()['response'])

In [ ]:
def chat_with_ollama(user_message):
    data = {
        "model": LLM_MODEL,
        "prompt": "Please identify the country of the affiliation. If the country is known, return the ISO country code (2-letter, e.g., US, CN). If the country is unknown or not confident, return 'Unknown'. Only return the country code, no explanation: {user_message}" + user_message,
        "stream": False
    }
    try:
        response = requests.post(generate_url, json=data)
        return response.json()['response']
    except requests.RequestException as e:
        return f"Request failed: {e}"

In [ ]:
L_to_search = list(set(geo_db.affiliationame.tolist()))

with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    ollama_results = list(tqdm(executor.map(chat_with_ollama, L_to_search), total=len(L_to_search)))

In [ ]:
df_result = pd.DataFrame({'affiliationame': L_to_search, 'ctry': ollama_results})
df_result

In [ ]:
country_abbr_db = pd.read_csv(dataset_config['path_other'] + 'ctry_country.csv')
country_abbr_db

In [ ]:
result_merged = df_result.merge(country_abbr_db, on='ctry', how='left')
result_merged

In [ ]:
result_recognized = result_merged[~result_merged.country.isna()][['affiliationame', 'country']]
result_recognized

In [ ]:
result_not_recognized = result_merged[result_merged.country.isna()][['affiliationame']]
result_not_recognized

In [ ]:
full_merged = to_search.merge(result_recognized, on='affiliationame', how='left')
full_merged

In [ ]:
data_recognized = full_merged[~full_merged.country.isna()]
data_recognized

In [ ]:
data_recognized.to_parquet(dataset_config['path_processed'] + 'CN_CN/POST2_recognized_by_LLM.parquet')

In [ ]:
data_not_recognized = full_merged[full_merged.country.isna()].drop(columns=['country'])
data_not_recognized

In [ ]:
data_not_recognized.to_parquet(dataset_config['path_processed'] + 'CN_CN/POST2_not_recognized.parquet')